In [14]:
import pandas as pd
import numpy as np


In [15]:
# ---------Load the dataset---------
df= pd.read_csv(r"C:/Users/ELCOT/Sample - Superstore.csv",encoding = 'latin-1')
# ---------Data Inspection-----------
# View first 5 rows
print(df.head())
# Check the dataset shape(rows,columns)
print(df.shape)
# Check data types and null values
df.info()
# Check column names
df.columns

   Row ID        Order ID  Order Date   Ship Date       Ship Mode Customer ID  \
0       1  CA-2016-152156  11-08-2016  11-11-2016    Second Class    CG-12520   
1       2  CA-2016-152156  11-08-2016  11-11-2016    Second Class    CG-12520   
2       3  CA-2016-138688  06-12-2016   6/16/2016    Second Class    DV-13045   
3       4  US-2015-108966  10-11-2015  10/18/2015  Standard Class    SO-20335   
4       5  US-2015-108966  10-11-2015  10/18/2015  Standard Class    SO-20335   

     Customer Name    Segment        Country             City  ...  \
0      Claire Gute   Consumer  United States        Henderson  ...   
1      Claire Gute   Consumer  United States        Henderson  ...   
2  Darrin Van Huff  Corporate  United States      Los Angeles  ...   
3   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   
4   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   

  Postal Code  Region       Product ID         Category Sub-Category  \
0       42420   Sout

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='object')

In [16]:
# ---------Column Cleaning---------
# Remove leading and trailing spaces from column names
df.columns = df.columns.str.strip() 
# Replace spaces between words with underscore
df.columns = df.columns.str.replace(' ','_')
# Convert all column names to lowercase
df.columns = df.columns.str.lower()
# Used to remove the special characters
df.columns = df.columns.str.replace('[^a-zA-Z0-9_]','',regex = True)

In [17]:
df.columns

Index(['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode',
       'customer_id', 'customer_name', 'segment', 'country', 'city', 'state',
       'postal_code', 'region', 'product_id', 'category', 'subcategory',
       'product_name', 'sales', 'quantity', 'discount', 'profit'],
      dtype='object')

In [18]:
# ---------Handling Missing Values---------
# Check missing values
df.isnull().sum()

row_id           0
order_id         0
order_date       0
ship_date        0
ship_mode        0
customer_id      0
customer_name    0
segment          0
country          0
city             0
state            0
postal_code      0
region           0
product_id       0
category         0
subcategory      0
product_name     0
sales            0
quantity         0
discount         0
profit           0
dtype: int64

In [50]:
# ---------Removing Duplicates---------
num_duplicates = int(df.duplicated().sum())
print(num_duplicates)

0


In [19]:
# ================= ORDER DATE CLEANING =================

# backup
df['original_order_date'] = df['order_date']

# Step 1: automatic parsing
df['order_date'] = pd.to_datetime(
    df['original_order_date'],
    errors='coerce'
)
# converts only failed rows using another format - select only rows where mask = True 
# fix remaining dataset using another format
# Step 2: find failed rows
mask = df['order_date'].isna()

# Step 3: fix remaining using dayfirst
df.loc[mask, 'order_date'] = pd.to_datetime(
    df.loc[mask, 'original_order_date'],
    errors='coerce',
    dayfirst=True
)

# check
print(df['order_date'].isna().sum())

0


C:\Users\ELCOT\AppData\Local\Temp\ipykernel_8812\757860474.py:17: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df.loc[mask, 'order_date'] = pd.to_datetime(


In [20]:
# ================= SHIP DATE CLEANING =================
# backup
df['original_ship_date'] = df['ship_date']

# Step 1: automatic parsing
df['ship_date'] = pd.to_datetime(
    df['original_ship_date'],
    errors='coerce'
)
# converts only failed rows using another format - select only rows where mask = True 
# fix remaining dataset using another format
# Step 2: find failed rows
mask = df['ship_date'].isna()

# Step 3: fix remaining using dayfirst
df.loc[mask, 'ship_date'] = pd.to_datetime(
    df.loc[mask, 'original_ship_date'],
    errors='coerce',
    dayfirst=True
)

# check
print(df['ship_date'].isna().sum())

0


C:\Users\ELCOT\AppData\Local\Temp\ipykernel_8812\1121678876.py:16: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df.loc[mask, 'ship_date'] = pd.to_datetime(


In [21]:
df['ship_date']

0      2016-11-11
1      2016-11-11
2      2016-06-16
3      2015-10-18
4      2015-10-18
          ...    
9989   2014-01-23
9990   2017-03-03
9991   2017-03-03
9992   2017-03-03
9993   2017-05-09
Name: ship_date, Length: 9994, dtype: datetime64[ns]

In [22]:
df['order_date']

0      2016-11-08
1      2016-11-08
2      2016-06-12
3      2015-10-11
4      2015-10-11
          ...    
9989   2014-01-21
9990   2017-02-26
9991   2017-02-26
9992   2017-02-26
9993   2017-05-04
Name: order_date, Length: 9994, dtype: datetime64[ns]

In [23]:
#feature engineering creating new useful columns from existing data
# Calculate delivery time in days
# Difference between ship_date and order_date
df['delivery_days'] = (df['ship_date'] - df['order_date']).dt.days
# trend analysis
df['order_year'] = df['order_date'].dt.year
# seasonal trends
df['order_month'] = df['order_date'].dt.month
df['order_day'] = df['order_date'].dt.day
# profit ratio - loss making products
df['profit_ratio'] = df['profit'] / df['sales']

In [24]:
df['delivery_days']

0       3
1       3
2       4
3       7
4       7
       ..
9989    2
9990    5
9991    5
9992    5
9993    5
Name: delivery_days, Length: 9994, dtype: int64

In [25]:
# Removed the backup columns - axis = 1 - columns, 0- rows
df.drop(['original_order_date','original_ship_date'], axis = 1, inplace = True)

In [26]:
df.dtypes

row_id                    int64
order_id                 object
order_date       datetime64[ns]
ship_date        datetime64[ns]
ship_mode                object
customer_id              object
customer_name            object
segment                  object
country                  object
city                     object
state                    object
postal_code               int64
region                   object
product_id               object
category                 object
subcategory              object
product_name             object
sales                   float64
quantity                  int64
discount                float64
profit                  float64
delivery_days             int64
order_year                int32
order_month               int32
order_day                 int32
profit_ratio            float64
dtype: object

In [23]:
# category dtype  - optimized text with few unique values - columns used when limited number of repaeated values
categorical_col = ['ship_mode','segment','region','category','subcategory']
for col in categorical_col:
    df[col] = df[col].astype('category')

df['postal_code'] = df['postal_code'].astype(str)    
df['city'] = df['city'].astype('object')
df['state'] = df['state'].astype('object')    

In [24]:
df.dtypes

row_id                    int64
order_id                 object
order_date       datetime64[ns]
ship_date        datetime64[ns]
ship_mode              category
customer_id              object
customer_name            object
segment                category
country                  object
city                     object
state                    object
postal_code              object
region                 category
product_id               object
category               category
subcategory            category
product_name             object
sales                   float64
quantity                  int64
discount                float64
profit                  float64
delivery_days             int64
dtype: object

In [27]:
# Outliers checking process
df[['sales','profit','quantity']].describe()

,sales,profit,quantity
count,9994.000000,9994.000000,9994.000000
mean,229.858001,28.656896,3.789574
std,623.245101,234.260108,2.225110
min,0.444000,-6599.978000,1.000000
25%,17.280000,1.728750,2.000000
50%,54.490000,8.666500,3.000000
75%,209.940000,29.364000,5.000000
max,22638.480000,8399.976000,14.000000


In [28]:
# Using IQR method is very effective
cols = ['sales','profit','quantity']
for column in cols:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    print(f"{column} outliers:", df[(df[column] < lower) | (df[column] > upper)].shape)

# explanation 1167 - this much values are  outside the normal range 22 - columns , profit - (-6000 loss, 8000 huge profit)

sales outliers: (1167, 26)
profit outliers: (1881, 26)
quantity outliers: (170, 26)


In [32]:
# comparing with total data

print("sales: " ,1167/9994) # 11% some high value orders
print("profit: ", 1881/9994) # 19% Loss + high profit
print("quantity: ",170/9994)  # 1.7% small variation



sales:  0.11677006203722233
profit:  0.188212927756654
quantity:  0.017010206123674206


In [33]:
#check cardiality helps to decide category usage
df.nunique().sort_values()

country             1
segment             3
category            3
ship_mode           4
region              4
order_year          4
delivery_days       8
discount           12
order_month        12
quantity           14
subcategory        17
order_day          31
state              49
city              531
profit_ratio      572
postal_code       631
customer_name     793
customer_id       793
order_date       1237
ship_date        1334
product_name     1850
product_id       1862
order_id         5009
sales            5825
profit           7287
row_id           9994
dtype: int64

In [33]:
df.columns


Index(['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode',
       'customer_id', 'customer_name', 'segment', 'country', 'city', 'state',
       'postal_code', 'region', 'product_id', 'category', 'subcategory',
       'product_name', 'sales', 'quantity', 'discount', 'profit',
       'delivery_days'],
      dtype='object')

In [34]:
# Validation check
print(df.dtypes)
print(df.isnull().sum())
print(df.duplicated().sum())

row_id                    int64
order_id                 object
order_date       datetime64[ns]
ship_date        datetime64[ns]
ship_mode                object
customer_id              object
customer_name            object
segment                  object
country                  object
city                     object
state                    object
postal_code               int64
region                   object
product_id               object
category                 object
subcategory              object
product_name             object
sales                   float64
quantity                  int64
discount                float64
profit                  float64
delivery_days             int64
order_year                int32
order_month               int32
order_day                 int32
profit_ratio            float64
dtype: object
row_id           0
order_id         0
order_date       0
ship_date        0
ship_mode        0
customer_id      0
customer_name    0
segment          0
co

In [35]:
# Save cleaned dataset
df.to_csv("cleaned_superstore.csv", index = False)